In [ ]:
import struct
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self):
        super(LSTMModel, self).__init__()
        self.layers = nn.ModuleList()
    
    def layer_maker(self, type, input_size, hidden_size, num_layers):
        if type == 'LSTM':
            return nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        elif type == 'GRU':
            return nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        elif type == 'RNN':
            return nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        elif type == 'linear':
            return nn.Linear(input_size, hidden_size)
        else:
            raise ValueError(f"Unsupported layer type: {type}")
    
    def create(self, x, struct: list[dict]):
        self.validate_struct(struct, expected_input_size=x.size(-1))
        for layer in struct:
            self.layers.append(
                self.layer_maker(
                    type=layer["type"], 
                    input_size=layer["input_size"], 
                    hidden_size=layer.get("hidden_size"), 
                    num_layers=layer.get("num_layers")
                )
            )
        return struct, self.layers
    def forward(self, x):
        lstm_outputs_processed = False
        
        for i, layer in enumerate(self.layers):
            if isinstance(layer, (nn.LSTM, nn.GRU, nn.RNN)):
                x, _ = layer(x)
                
            elif isinstance(layer, nn.Linear):
                if not lstm_outputs_processed and x.dim() == 3:
                    x = x[:, -1, :]
                    lstm_outputs_processed = True
    
                x = layer(x)  
        
        return x
    
    def validate_struct(self, struct: list[dict], expected_input_size: int):
        prev_output_size = expected_input_size

        for i, layer in enumerate(struct):
            required = {"type", "input_size", "output_size"}
            missing = required - layer.keys()
            if missing:
                raise ValueError(f"Layer {i} missing keys: {missing}")

            if layer["input_size"] != prev_output_size:
                raise ValueError(
                    f"Layer {i} input_size {layer['input_size']} "
                    f"does not match previous output_size {prev_output_size}"
                )

            if layer["type"] in {"LSTM", "GRU", "RNN"}:
                if "hidden_size" not in layer or "num_layers" not in layer:
                    raise ValueError(f"Layer {i} missing hidden_size or num_layers")

                if layer["output_size"] != layer["hidden_size"]:
                    raise ValueError(
                        f"Layer {i} output_size must equal hidden_size for RNN layers"
                    )

            prev_output_size = layer["output_size"]



def train_model(model, X_train, y_train, X_test=None, y_test=None, epochs=50, batch_size=32, lr=0.001):    

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    train_losses = []
    test_losses = []
    
    num_batches = len(X_train) // batch_size
    
    print(f"\nTraining Configuration:")
    print(f"  Epochs: {epochs}")
    print(f"  Batch size: {batch_size}")
    print(f"  Learning rate: {lr}")
    print(f"  Batches per epoch: {num_batches}")
    print(f"  Training samples: {len(X_train)}")
    if X_test is not None:
        print(f"  Test samples: {len(X_test)}")
    print("=" * 70)
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        
        # Shuffle training data
        indices = torch.randperm(len(X_train))
        X_train_shuffled = X_train[indices]
        y_train_shuffled = y_train[indices]
        
        # Mini-batch training
        for batch_idx in range(num_batches):
            start_idx = batch_idx * batch_size
            end_idx = start_idx + batch_size
            
            # Get batch
            batch_X = X_train_shuffled[start_idx:end_idx]
            batch_y = y_train_shuffled[start_idx:end_idx]
            
            # Forward pass
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        # Calculate average training loss
        avg_train_loss = epoch_loss / num_batches
        train_losses.append(avg_train_loss)
        
        # Evaluate on test set if provided
        test_loss = None
        if X_test is not None and y_test is not None:
            model.eval()
            with torch.no_grad():
                test_predictions = model(X_test)
                test_loss = criterion(test_predictions, y_test).item()
                test_losses.append(test_loss)
        
        # Print progress (every epoch or every 5 epochs)
        if epochs <= 20 or (epoch + 1) % 5 == 0 or epoch == 0:
            if test_loss is not None:
                print(f"Epoch [{epoch+1:3d}/{epochs}] - Train Loss: {avg_train_loss:.4f} - Test Loss: {test_loss:.4f}")
            else:
                print(f"Epoch [{epoch+1:3d}/{epochs}] - Train Loss: {avg_train_loss:.4f}")
    
    print("=" * 70)
    print("Training Complete!")
    print(f"Final Train Loss: {train_losses[-1]:.4f}")
    if test_losses:
        print(f"Final Test Loss: {test_losses[-1]:.4f}")
    print("=" * 70)
    
    if test_losses:
        return train_losses, test_losses
    else:
        return train_losses

def evaluate_model(model, X_test, y_test, mean_y, std_y):
    """Evaluate model and show sample predictions."""
    model.eval()
    
    with torch.no_grad():
        predictions_norm = model(X_test)
    
    # Denormalize predictions and targets
    predictions = predictions_norm * std_y + mean_y
    targets = y_test * std_y + mean_y
    
    # Calculate metrics
    mse = ((predictions - targets) ** 2).mean().item()
    mae = (predictions - targets).abs().mean().item()
    
    print("\nEvaluation Results:")
    print("=" * 70)
    print(f"Test MSE: {mse:.2f}")
    print(f"Test MAE: {mae:.2f}")
    print(f"\nSample Predictions (denormalized to actual prices):")
    print("-" * 70)
    print(f"{'Predicted':>12} | {'Actual':>12} | {'Difference':>12}")
    print("-" * 70)
    
    for i in range(min(10, len(predictions))):
        pred = predictions[i].item()
        actual = targets[i].item()
        diff = abs(pred - actual)
        print(f"{pred:12.2f} | {actual:12.2f} | {diff:12.2f}")




    

In [ ]:
from start import StockDataManager
model = LSTMModel()


#evaluate_model(model, generate_sample_stock_data(), torch.rand(32, 1), mean_y=250, std_y=100)


print("\n### EXAMPLE 1: Fetch and Prepare Data ###\n")
    
manager = StockDataManager(save_dir='stock_data')
    
data = manager.prepare_for_training(
        symbols=['AAPL', 'MSFT', 'GOOGL', 'TSLA', 'AMZN'],
        seq_length=60,           # Use 60 days to predict next day
        prediction_days=1,       # Predict 1 day ahead
        total_days=365,          # Fetch 1 year of data
        train_ratio=0.8,         # 80% train, 20% test
        normalize=True,          # Normalize data
        save=True                # Save to disk
    )

In [27]:
# Get data
X_train = data['X_train']
y_train = data['y_train']
X_test = data['X_test']
y_test = data['y_test']

# Fix y shapes
y_train = y_train.squeeze()
y_test = y_test.squeeze()

if y_train.dim() == 1:
    y_train = y_train.unsqueeze(1)
if y_test.dim() == 1:
    y_test = y_test.unsqueeze(1)

# Verify shapes
print("Data shapes:")
print(f"X_train: {X_train.shape}")  # Should be [920, 60, 5]
print(f"y_train: {y_train.shape}")  # Should be [920, 1]
print(f"X_test: {X_test.shape}")    # Should be [230, 60, 5]
print(f"y_test: {y_test.shape}")    # Should be [230, 1]

# Define structure
struct = [
    {
        "type": "LSTM",
        "input_size": 5,
        "hidden_size": 32,
        "num_layers": 1,
        "output_size": 32
    },
    {
        "type": "linear",
        "input_size": 32,
        "hidden_size": 16,
        "output_size": 16
    },
    {
        "type": "linear",
        "input_size": 16,
        "hidden_size": 1,
        "output_size": 1
    }
]

# Create NEW model (don't reuse old one!)
model = LSTMModel()  # Fresh instance
struct_used, layers = model.create(X_train, struct)

print(f"\nModel created with {len(model.layers)} layers")

# Train
train_losses, test_losses = train_model(
    model, 
    X_train, 
    y_train, 
    X_test, 
    y_test,
    epochs=500, 
    batch_size=48, 
    lr=0.00001
)

Data shapes:
X_train: torch.Size([920, 60, 5])
y_train: torch.Size([920, 1])
X_test: torch.Size([230, 60, 5])
y_test: torch.Size([230, 1])

Model created with 3 layers

Training Configuration:
  Epochs: 500
  Batch size: 48
  Learning rate: 1e-05
  Batches per epoch: 19
  Training samples: 920
  Test samples: 230
Epoch [  1/500] - Train Loss: 1.1379 - Test Loss: 0.6747
Epoch [  5/500] - Train Loss: 1.1259 - Test Loss: 0.6674
Epoch [ 10/500] - Train Loss: 1.0977 - Test Loss: 0.6587
Epoch [ 15/500] - Train Loss: 1.0709 - Test Loss: 0.6499
Epoch [ 20/500] - Train Loss: 1.0526 - Test Loss: 0.6410
Epoch [ 25/500] - Train Loss: 1.0237 - Test Loss: 0.6316
Epoch [ 30/500] - Train Loss: 1.0024 - Test Loss: 0.6218
Epoch [ 35/500] - Train Loss: 0.9753 - Test Loss: 0.6114
Epoch [ 40/500] - Train Loss: 0.9441 - Test Loss: 0.5999
Epoch [ 45/500] - Train Loss: 0.9149 - Test Loss: 0.5878
Epoch [ 50/500] - Train Loss: 0.8868 - Test Loss: 0.5751
Epoch [ 55/500] - Train Loss: 0.8483 - Test Loss: 0.5615
E

In [5]:
import pickle
import torch


def print_stock_data(filepath='stock_data/stock_data.pkl'):
    """Print contents of saved stock data file."""
    
    # Load the data
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    
    print("=" * 70)
    print("STOCK DATA FILE CONTENTS")
    print("=" * 70)
    
    # Print symbols
    print(f"\nStocks: {', '.join(data['symbols'])}")
    
    # Print shapes
    print(f"\nData Shapes:")
    print(f"  X (inputs):  {data['X'].shape}")
    print(f"  y (targets): {data['y'].shape}")
    
    # Print a sample
    print(f"\n{'='*70}")
    print("SAMPLE DATA (First Sequence)")
    print("=" * 70)
    
    X_sample = data['X'][0]
    y_sample = data['y'][0]
    
    # Denormalize
    if data['mean_X'] is not None:
        X_sample = X_sample * data['std_X'] + data['mean_X']
        y_sample = y_sample * data['std_y'] + data['mean_y']
    
    print(f"\nTarget Price: ${y_sample.item():.2f}")
    print(f"\nInput Sequence (60 days of OHLCV):")
    print("-" * 70)
    print(f"{'Day':>4} | {'Open':>10} | {'High':>10} | {'Low':>10} | {'Close':>10} | {'Volume':>12}")
    print("-" * 70)
    
    # Print first 10 days
    for i in range(min(10, len(X_sample))):
        open_p, high, low, close, volume = X_sample[i]
        print(f"{i+1:4d} | ${open_p:9.2f} | ${high:9.2f} | ${low:9.2f} | ${close:9.2f} | {volume:12.0f}")
    
    if len(X_sample) > 20:
        print("  ...")
        # Print last 10 days
        for i in range(len(X_sample) - 10, len(X_sample)):
            open_p, high, low, close, volume = X_sample[i]
            print(f"{i+1:4d} | ${open_p:9.2f} | ${high:9.2f} | ${low:9.2f} | ${close:9.2f} | {volume:12.0f}")
    elif len(X_sample) > 10:
        for i in range(10, len(X_sample)):
            open_p, high, low, close, volume = X_sample[i]
            print(f"{i+1:4d} | ${open_p:9.2f} | ${high:9.2f} | ${low:9.2f} | ${close:9.2f} | {volume:12.0f}")
    
    print("-" * 70)
    print(f"{'→':>4} | {'':>10} | {'':>10} | {'':>10} | ${y_sample.item():9.2f} | {'(Target)':>12}")
    print("=" * 70)
    
    # Print statistics
    print(f"\nNormalization Parameters:")
    if data['mean_X'] is not None:
        print(f"  mean_X: {data['mean_X']}")
        print(f"  std_X:  {data['std_X']}")
        print(f"  mean_y: {data['mean_y'].item():.2f}")
        print(f"  std_y:  {data['std_y'].item():.2f}")
    else:
        print("  Not normalized")
    
    print("\n" + "=" * 70)


# Usage
if __name__ == "__main__":
    print_stock_data('stock_data/stock_data.pkl')

STOCK DATA FILE CONTENTS

Stocks: AAPL, MSFT, GOOGL, TSLA, AMZN

Data Shapes:
  X (inputs):  torch.Size([1150, 60, 5])
  y (targets): torch.Size([1150, 1, 1])

SAMPLE DATA (First Sequence)

Target Price: $23935.33

Input Sequence (60 days of OHLCV):
----------------------------------------------------------------------
 Day |       Open |       High |        Low |      Close |       Volume
----------------------------------------------------------------------
   1 | $ 27852.04 | $ 28488.27 | $ 27525.91 | $ 28183.55 | 1760081355997184
   2 | $ 28081.23 | $ 28538.25 | $ 27303.49 | $ 27586.15 | 1945756281864192
   3 | $ 27567.76 | $ 28285.05 | $ 27232.62 | $ 27777.58 | 2086552154931200
   4 | $ 27627.26 | $ 28618.21 | $ 27079.98 | $ 28294.67 | 5054944042287104
   5 | $ 28368.83 | $ 28690.39 | $ 27926.06 | $ 28380.49 | 1400308722827264
   6 | $ 28448.17 | $ 28974.69 | $ 28126.68 | $ 28702.84 | 796297405661184
   7 | $ 28745.67 | $ 29184.57 | $ 28381.81 | $ 28793.05 | 933467252588544
   8 |